# 02 Entity Extraction With Qwen

Run dictionary/rule-first entity extraction with full local Qwen augmentation, optional diagnostic preflight, command timeouts, and resume state.

Partition this notebook by strategy. Keep `SELECTED_SOURCES` in the shared contract for bookkeeping, but do not split entity extraction by source in Kaggle.

In [ ]:
import json, os, re, shutil, signal, subprocess, sys, time, zipfile
from pathlib import Path

CORPUS_SLUG = 'tuvi-golden-corpus'
SCRIPTS_SLUG = 'tuvi-battu-scripts'

ALL_STRATEGIES = ['chunk_fixed_512', 'chunk_structure_parent_child', 'chunk_semantic_embedding_bge_m3']
SOURCE_IDS = ['TVKL', 'TVNL', 'TVHS', 'TVGM']
RUN_TAG = 'part_a'
PARTITION_MODE = 'strategy'
SELECTED_STRATEGIES = ['chunk_fixed_512', 'chunk_structure_parent_child']
SELECTED_SOURCES = list(SOURCE_IDS)
LLM_AUGMENTATION = 'on'
LOCAL_LLM_MODEL = 'Qwen/Qwen2.5-7B-Instruct'
LOCAL_LLM_QUANTIZATION = 'none'
LOCAL_LLM_DEVICE = 'auto-cuda'
LOCAL_LLM_MAX_NEW_TOKENS = 1024
LOCAL_LLM_MAX_JSON_RETRIES = 2
LLM_PREFLIGHT_CHUNK_LIMIT = 0  # set to 1 only for a diagnostic run; 0 avoids loading Qwen twice
LLM_PREFLIGHT_TIMEOUT_SECONDS = 900
ENTITY_MAX_RUNTIME_SECONDS = 36000
ENTITY_COMMAND_TIMEOUT_SECONDS = 39600
PREVIOUS_OUTPUT_SLUGS = []  # required notebook 01 chunks; e.g. ['w3-local-outputs/w3_local_outputs_01_part_a']
PREVIOUS_ENTITY_OUTPUT_SLUGS = []  # optional notebook 02 partial resume; e.g. ['w3-local-outputs/w3_local_outputs_02_part_a']
INPUT_ROOT_CANDIDATES = [
    Path('/kaggle/input/datasets/dinhbaobao'),
    Path('/kaggle/input'),
]

def resolve_input_root() -> Path:
    for candidate in INPUT_ROOT_CANDIDATES:
        if candidate.exists():
            return candidate
    return Path('/kaggle/input')

def resolve_input_path(value: str) -> Path:
    path = Path(value)
    if path.is_absolute():
        return path
    return INPUT_ROOT / path

INPUT_ROOT = resolve_input_root()
CORPUS_ROOT = resolve_input_path(CORPUS_SLUG)
SCRIPTS_ROOT = resolve_input_path(SCRIPTS_SLUG)
CORPUS_DIR = CORPUS_ROOT / 'benchmark' / 'tuvi_golden_dataset'
SCRIPTS_INPUT_DIR = SCRIPTS_ROOT
RUNTIME_SCRIPTS_DIR = Path('/kaggle/working/tuvi_battu_scripts_runtime')
SCRIPTS_DIR = RUNTIME_SCRIPTS_DIR
if not CORPUS_DIR.exists():
    CORPUS_DIR = Path.cwd() / 'benchmark' / 'tuvi_golden_dataset'
if not SCRIPTS_INPUT_DIR.exists():
    SCRIPTS_INPUT_DIR = Path.cwd()

ACTIVE_STRATEGIES = [item for item in ALL_STRATEGIES if item in SELECTED_STRATEGIES]
if not ACTIVE_STRATEGIES:
    raise ValueError('No active strategies selected')

OUTPUT_ROOT = Path('/kaggle/working/w3_local_outputs')
PREFLIGHT_ROOT = Path('/kaggle/working/w3_preflight')
REPORTS = OUTPUT_ROOT / 'reports'
STATE = OUTPUT_ROOT / 'state'
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
PREFLIGHT_ROOT.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)
STATE.mkdir(parents=True, exist_ok=True)

def copy_tree_contents(source_dir: Path, destination_dir: Path) -> None:
    for child in source_dir.iterdir():
        target = destination_dir / child.name
        if child.is_dir():
            shutil.copytree(child, target, dirs_exist_ok=True)
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(child, target)

def find_unpacked_artifact_roots(input_root: Path, expected_dir_name: str) -> list[Path]:
    named_dirs = []
    if input_root.is_dir() and input_root.name == expected_dir_name:
        named_dirs.append(input_root)
    if input_root.is_dir():
        named_dirs.extend(path for path in input_root.rglob(expected_dir_name) if path.is_dir())
    search_dirs = named_dirs or ([input_root] if input_root.is_dir() else [])
    artifact_roots = []
    for directory in search_dirs:
        if (directory / 'chunks').is_dir():
            artifact_roots.append(directory)
        artifact_roots.extend(path.parent for path in directory.rglob('chunks') if path.is_dir())
    unique_roots = []
    seen = set()
    for root in artifact_roots:
        resolved = root.resolve()
        if resolved not in seen:
            seen.add(resolved)
            unique_roots.append(root)
    return unique_roots

def restore_previous_outputs(slugs, expected_zip_name, expected_dir_name, *, required: bool):
    restored = []
    missing_messages = []
    search_roots = [resolve_input_path(slug) for slug in slugs] if slugs else [INPUT_ROOT]
    for input_root in search_roots:
        if not input_root.exists():
            missing_messages.append(f'Previous output dataset not attached: {input_root}')
            continue
        zip_files = sorted(path for path in input_root.rglob(expected_zip_name) if path.is_file())
        artifact_roots = find_unpacked_artifact_roots(input_root, expected_dir_name)
        if zip_files:
            for zip_path in zip_files:
                print('Restoring previous output zip =', zip_path)
                with zipfile.ZipFile(zip_path) as archive:
                    archive.extractall(OUTPUT_ROOT)
                restored.append(str(zip_path))
            continue
        if artifact_roots:
            selected_root = artifact_roots[0]
            print('Restoring previous output folder =', selected_root)
            copy_tree_contents(selected_root, OUTPUT_ROOT)
            restored.append(str(selected_root))
            continue
        missing_messages.append(f'No {expected_zip_name} or {expected_dir_name} found under {input_root}')
    if required and not restored:
        raise FileNotFoundError('; '.join(missing_messages) or f'Missing required artifact {expected_dir_name}')
    return restored

def apply_text_patch(path: Path, replacements: list[tuple[str, str]]) -> int:
    text = path.read_text(encoding='utf-8')
    patch_count = 0
    for old, new in replacements:
        if old in text:
            text = text.replace(old, new)
            patch_count += 1
    path.write_text(text, encoding='utf-8')
    return patch_count

def apply_notebook_runtime_hotfixes(runtime_root: Path) -> dict[str, int]:
    patch_counts = {}
    extract_path = runtime_root / 'scripts' / 'extract_entities.py'
    local_llm_path = runtime_root / 'scripts' / 'local_llm.py'
    safe_entities_line = 'entities = payload if isinstance(payload, list) else (payload.get("entities", []) if isinstance(payload, dict) else [])'
    patch_counts['extract_entities_payload_list'] = apply_text_patch(
        extract_path,
        [('entities = payload.get("entities", payload if isinstance(payload, list) else [])', safe_entities_line)],
    )

    local_text = local_llm_path.read_text(encoding='utf-8')
    hotfix_parse_json_payload = r'''def parse_json_payload(text: str) -> Any:
    def _extract_candidate(value: str) -> str:
        stripped = value.strip()
        fenced = re.match(r"^```(?:json)?\s*(.*?)\s*```$", stripped, flags=re.S | re.I)
        if fenced:
            stripped = fenced.group(1).strip()
        start_positions = [pos for pos in (stripped.find("{"), stripped.find("[")) if pos >= 0]
        if not start_positions:
            return stripped
        start = min(start_positions)
        open_char = stripped[start]
        close_char = "}" if open_char == "{" else "]"
        depth = 0
        in_string = False
        escape = False
        for index in range(start, len(stripped)):
            char = stripped[index]
            if escape:
                escape = False
                continue
            if char == "\\":
                escape = True
                continue
            if char == '"':
                in_string = not in_string
                continue
            if in_string:
                continue
            if char == open_char:
                depth += 1
            elif char == close_char:
                depth -= 1
                if depth == 0:
                    return stripped[start:index + 1]
        return stripped[start:]

    def _repair(candidate: str) -> str:
        repaired = re.sub(r",\s*([}\]])", r"\1", candidate.strip())
        for open_char, close_char in (("{", "}"), ("[", "]")):
            delta = repaired.count(open_char) - repaired.count(close_char)
            if delta > 0:
                repaired += close_char * delta
        return repaired

    candidate = _extract_candidate(text)
    try:
        return json.loads(candidate)
    except json.JSONDecodeError:
        return json.loads(_repair(candidate))
'''
    local_text, parser_patch_count = re.subn(
        r'def parse_json_payload\(text: str\) -> Any:\n.*?\n\nclass LocalQwenJsonClient:',
        lambda _: hotfix_parse_json_payload + '\n\nclass LocalQwenJsonClient:',
        local_text,
        count=1,
        flags=re.S,
    )
    if 'model_kwargs["torch_dtype"]' in local_text:
        local_text = local_text.replace('model_kwargs["torch_dtype"]', 'model_kwargs["dtype"]')
        parser_patch_count += 1
    local_llm_path.write_text(local_text, encoding='utf-8')
    patch_counts['local_llm_json_parser'] = parser_patch_count
    return patch_counts

def prepare_runtime_scripts() -> Path:
    if RUNTIME_SCRIPTS_DIR.exists():
        shutil.rmtree(RUNTIME_SCRIPTS_DIR)
    shutil.copytree(SCRIPTS_INPUT_DIR, RUNTIME_SCRIPTS_DIR)
    patch_counts = apply_notebook_runtime_hotfixes(RUNTIME_SCRIPTS_DIR)
    print('Notebook runtime scripts prepared =', RUNTIME_SCRIPTS_DIR)
    print('Notebook runtime hotfixes =', patch_counts)
    return RUNTIME_SCRIPTS_DIR

def terminate_process(process: subprocess.Popen) -> None:
    if process.poll() is not None:
        return
    if os.name == 'posix':
        os.killpg(process.pid, signal.SIGTERM)
    else:
        process.terminate()
    try:
        process.wait(timeout=30)
    except subprocess.TimeoutExpired:
        if os.name == 'posix':
            os.killpg(process.pid, signal.SIGKILL)
        else:
            process.kill()
        process.wait(timeout=30)

def is_parent_chunk_record(record: dict, strategy: str) -> bool:
    chunk_type = str(record.get('chunk_type') or '')
    chunk_id = str(record.get('chunk_id') or '')
    return strategy == 'chunk_structure_parent_child' and (chunk_type == 'parent' or '_parent_' in chunk_id)

def count_processable_chunks(strategy: str, limit_chunks: int | None = None) -> int:
    count = 0
    chunk_dir = OUTPUT_ROOT / 'chunks' / strategy
    for path in sorted(chunk_dir.glob('*.jsonl')):
        with path.open('r', encoding='utf-8') as handle:
            for line in handle:
                if not line.strip():
                    continue
                record = json.loads(line)
                if record.get('chunk_strategy_id') != strategy:
                    continue
                if is_parent_chunk_record(record, strategy):
                    continue
                count += 1
                if limit_chunks is not None and count >= limit_chunks:
                    return count
    return count

def completed_state_count(state_path: Path | None) -> int:
    if state_path is None or not state_path.exists():
        return 0
    try:
        state = json.loads(state_path.read_text(encoding='utf-8'))
    except Exception:
        return 0
    completed = state.get('completed_chunks') if isinstance(state, dict) else None
    return len(completed) if isinstance(completed, dict) else 0

def make_progress_bar(label: str, total: int, initial: int):
    try:
        from tqdm.auto import tqdm
    except Exception:
        return None
    bar = tqdm(total=total, initial=min(initial, total), desc=label, unit='chunk', dynamic_ncols=True)
    return bar

def run_checked(cmd: list[str], *, timeout_seconds: int, progress_label: str | None = None, state_path: Path | None = None, expected_total: int | None = None) -> None:
    start_new_session = os.name == 'posix'
    process = subprocess.Popen(cmd, cwd=SCRIPTS_DIR, start_new_session=start_new_session)
    started_at = time.monotonic()
    last_log_at = 0.0
    progress_bar = None
    last_completed = completed_state_count(state_path)
    if progress_label and expected_total:
        progress_bar = make_progress_bar(progress_label, expected_total, last_completed)
        if progress_bar is None:
            print(f'{progress_label}: {last_completed}/{expected_total} chunks completed', flush=True)
            last_log_at = time.monotonic()
    try:
        while True:
            try:
                return_code = process.wait(timeout=30)
                break
            except subprocess.TimeoutExpired:
                if time.monotonic() - started_at >= timeout_seconds:
                    terminate_process(process)
                    raise TimeoutError(f'Command timed out after {timeout_seconds}s: {cmd}')
                if progress_label and expected_total and state_path is not None:
                    completed = min(completed_state_count(state_path), expected_total)
                    if progress_bar is not None:
                        progress_bar.n = completed
                        progress_bar.set_postfix(elapsed_min=round((time.monotonic() - started_at) / 60, 1))
                        progress_bar.refresh()
                    elif completed != last_completed or time.monotonic() - last_log_at >= 60:
                        print(f'{progress_label}: {completed}/{expected_total} chunks completed, elapsed_min={round((time.monotonic() - started_at) / 60, 1)}', flush=True)
                        last_log_at = time.monotonic()
                    last_completed = completed
    except KeyboardInterrupt:
        terminate_process(process)
        raise
    finally:
        if progress_bar is not None:
            completed = min(completed_state_count(state_path), expected_total or progress_bar.total or 0)
            progress_bar.n = completed
            progress_bar.refresh()
            progress_bar.close()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, cmd)

SCRIPTS_DIR = prepare_runtime_scripts()

RESTORED_CHUNK_OUTPUTS = restore_previous_outputs(PREVIOUS_OUTPUT_SLUGS, f'w3_local_outputs_01_{RUN_TAG}.zip', f'w3_local_outputs_01_{RUN_TAG}', required=True)
RESTORED_ENTITY_OUTPUTS = restore_previous_outputs(PREVIOUS_ENTITY_OUTPUT_SLUGS, f'w3_local_outputs_02_{RUN_TAG}.zip', f'w3_local_outputs_02_{RUN_TAG}', required=bool(PREVIOUS_ENTITY_OUTPUT_SLUGS))
RESTORED_OUTPUTS = RESTORED_CHUNK_OUTPUTS + RESTORED_ENTITY_OUTPUTS
missing_chunks = [str(OUTPUT_ROOT / 'chunks' / strategy) for strategy in ACTIVE_STRATEGIES if not (OUTPUT_ROOT / 'chunks' / strategy).exists()]
if missing_chunks:
    raise FileNotFoundError('Missing chunk inputs. Attach notebook 01 artifact and set PREVIOUS_OUTPUT_SLUGS. Missing: ' + ', '.join(missing_chunks))

print('INPUT_ROOT =', INPUT_ROOT)
print('CORPUS_DIR =', CORPUS_DIR)
print('SCRIPTS_INPUT_DIR =', SCRIPTS_INPUT_DIR)
print('RUNTIME_SCRIPTS_DIR =', RUNTIME_SCRIPTS_DIR)
print('SCRIPTS_DIR =', SCRIPTS_DIR)
print('OUTPUT_ROOT =', OUTPUT_ROOT)
print('RESTORED_CHUNK_OUTPUTS =', RESTORED_CHUNK_OUTPUTS)
print('RESTORED_ENTITY_OUTPUTS =', RESTORED_ENTITY_OUTPUTS)
print('RESTORED_OUTPUTS =', RESTORED_OUTPUTS)
print(json.dumps({'run_tag': RUN_TAG, 'partition_mode': PARTITION_MODE, 'active_strategies': ACTIVE_STRATEGIES, 'selected_sources_recorded_only': SELECTED_SOURCES, 'llm_model': LOCAL_LLM_MODEL, 'llm_quantization': LOCAL_LLM_QUANTIZATION, 'llm_device': LOCAL_LLM_DEVICE, 'local_llm_max_json_retries': LOCAL_LLM_MAX_JSON_RETRIES, 'preflight_chunk_limit': LLM_PREFLIGHT_CHUNK_LIMIT, 'entity_max_runtime_seconds': ENTITY_MAX_RUNTIME_SECONDS}, ensure_ascii=False, indent=2))

In [ ]:
def build_entity_command(strategy: str, *, output_path: Path, review_path: Path, summary_path: Path, state_path: Path, resume: bool, limit_chunks: int | None = None) -> list[str]:
    cmd = [
        sys.executable, '-B', str(SCRIPTS_DIR / 'scripts' / 'extract_entities.py'),
        '--input', str(OUTPUT_ROOT / 'chunks' / strategy),
        '--output', str(output_path),
        '--chunking-strategy', strategy,
        '--review-output', str(review_path),
        '--partial-summary-output', str(summary_path),
        '--state-output', str(state_path),
        '--llm-augmentation', LLM_AUGMENTATION,
        '--llm-backend', 'local',
        '--model', LOCAL_LLM_MODEL,
        '--local-llm-quantization', LOCAL_LLM_QUANTIZATION,
        '--local-llm-max-new-tokens', str(LOCAL_LLM_MAX_NEW_TOKENS),
        '--local-llm-max-json-retries', str(LOCAL_LLM_MAX_JSON_RETRIES),
    ]
    if resume:
        cmd.append('--resume')
    if limit_chunks is not None:
        cmd += ['--limit-chunks', str(limit_chunks)]
    if LOCAL_LLM_DEVICE:
        cmd += ['--local-llm-device', LOCAL_LLM_DEVICE]
    if not limit_chunks and ENTITY_MAX_RUNTIME_SECONDS:
        cmd += ['--max-runtime-seconds', str(ENTITY_MAX_RUNTIME_SECONDS)]
    return cmd

for strategy in ACTIVE_STRATEGIES:
    if LLM_AUGMENTATION == 'on' and LLM_PREFLIGHT_CHUNK_LIMIT:
        preflight_state_path = PREFLIGHT_ROOT / f'{strategy}_entity_state_preflight.json'
        preflight_cmd = build_entity_command(
            strategy,
            output_path=PREFLIGHT_ROOT / f'{strategy}_entities_preflight.jsonl',
            review_path=PREFLIGHT_ROOT / f'{strategy}_entity_review_preflight.json',
            summary_path=PREFLIGHT_ROOT / f'{strategy}_entity_summary_preflight.json',
            state_path=preflight_state_path,
            resume=False,
            limit_chunks=LLM_PREFLIGHT_CHUNK_LIMIT,
        )
        print('Running Qwen preflight strategy =', strategy, 'chunk_limit =', LLM_PREFLIGHT_CHUNK_LIMIT)
        run_checked(
            preflight_cmd,
            timeout_seconds=LLM_PREFLIGHT_TIMEOUT_SECONDS,
            progress_label=f'preflight:{strategy}',
            state_path=preflight_state_path,
            expected_total=count_processable_chunks(strategy, limit_chunks=LLM_PREFLIGHT_CHUNK_LIMIT),
        )

    state_path = STATE / f'{strategy}_entity_state.json'
    cmd = build_entity_command(
        strategy,
        output_path=OUTPUT_ROOT / 'entities' / strategy / f'{strategy}_entities.jsonl',
        review_path=REPORTS / f'{strategy}_entity_review.json',
        summary_path=REPORTS / f'{strategy}_entity_summary.json',
        state_path=state_path,
        resume=True,
    )
    print('Running strategy =', strategy)
    run_checked(
        cmd,
        timeout_seconds=ENTITY_COMMAND_TIMEOUT_SECONDS,
        progress_label=strategy,
        state_path=state_path,
        expected_total=count_processable_chunks(strategy),
    )

print('Skipped strategies =', [item for item in ALL_STRATEGIES if item not in ACTIVE_STRATEGIES])

In [ ]:
for strategy in ACTIVE_STRATEGIES:
    summary_path = REPORTS / f'{strategy}_entity_summary.json'
    review_path = REPORTS / f'{strategy}_entity_review.json'
    state_path = STATE / f'{strategy}_entity_state.json'
    print(strategy)
    print('  summary =', summary_path.exists(), summary_path)
    print('  review  =', review_path.exists(), review_path)
    print('  state   =', state_path.exists(), state_path)
    if summary_path.exists():
        summary = json.loads(summary_path.read_text(encoding='utf-8'))
        print('  completed =', summary.get('completed'), 'stop_reason =', summary.get('stop_reason'))
        print('  processed =', summary.get('processed_chunk_count'), 'resume_skipped =', summary.get('resume_skipped_count'), 'errors =', summary.get('error_count'))
        print('  llm_calls =', summary.get('local_llm_call_count'), 'elapsed_seconds =', summary.get('elapsed_seconds'))
print('Skipped strategies =', [item for item in ALL_STRATEGIES if item not in ACTIVE_STRATEGIES])

In [ ]:
archive = shutil.make_archive(f'/kaggle/working/w3_local_outputs_02_{RUN_TAG}', 'zip', OUTPUT_ROOT)
print('Wrote', archive)
print('Attach or upload this zip as input for notebook 03, then set PREVIOUS_OUTPUT_SLUGS there.')